## Day 5 Concepts

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("Day05-Practice")\
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/23 10:40:02 WARN Utils: Your hostname, LF-00002297, resolves to a loopback address: 127.0.1.1; using 192.168.1.85 instead (on interface wlp0s20f3)
26/09/23 10:40:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/leapfrog/LakeHouse-Fifteen---Data-Engineering-Course/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/23 10:40:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark

In [10]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold" , -1)
spark.conf.set("spark.sql.adaptive.enabled" , False)

from pyspark.sql.functions import broadcast

#### Broadcast join

In [4]:
cust_data = [
    ("c1", "Alice"),
    ("c2", "Joey"),
    ("c3", "Rachel"),
    ("c4", "Chandelr"),
]

cust_column = ['c_id' , "name"]

cust_df = spark.createDataFrame(cust_data , cust_column)

order_data = [
    ( "c2" , 'Sandwhich' , 12.00),
    ( 'c3' , "Makeup" , 100.00),
    ('c4' , "Laptop", 200.00)  ,
    ('c2' , 'Pasta' , 5.00),
    ('c1' , "Bread" , 2.00 )
]

order_column = ['c_id' , 'items' , 'price']

order_df = spark.createDataFrame(order_data , order_column)


In [5]:
cust_df.show()

+----+--------+
|c_id|    name|
+----+--------+
|  c1|   Alice|
|  c2|    Joey|
|  c3|  Rachel|
|  c4|Chandelr|
+----+--------+



In [6]:
order_df.show()

+----+---------+-----+
|c_id|    items|price|
+----+---------+-----+
|  c2|Sandwhich| 12.0|
|  c3|   Makeup|100.0|
|  c4|   Laptop|200.0|
|  c2|    Pasta|  5.0|
|  c1|    Bread|  2.0|
+----+---------+-----+



In [ ]:
df_join_broad = order_df.join(broadcast(cust_df), order_df["c_id"] == cust_df['c_id'] ,'left') #Broadcast join

In [13]:
df_join_shuffle = order_df.join((cust_df), order_df["c_id"] == cust_df['c_id'] ,'left') #shuffle by default

In [15]:
df_join_shuffle.show()

+----+---------+-----+----+--------+
|c_id|    items|price|c_id|    name|
+----+---------+-----+----+--------+
|  c1|    Bread|  2.0|  c1|   Alice|
|  c4|   Laptop|200.0|  c4|Chandelr|
|  c3|   Makeup|100.0|  c3|  Rachel|
|  c2|Sandwhich| 12.0|  c2|    Joey|
|  c2|    Pasta|  5.0|  c2|    Joey|
+----+---------+-----+----+--------+



In [9]:
df_join_broad.show()

+----+---------+-----+----+--------+
|c_id|    items|price|c_id|    name|
+----+---------+-----+----+--------+
|  c2|Sandwhich| 12.0|  c2|    Joey|
|  c3|   Makeup|100.0|  c3|  Rachel|
|  c4|   Laptop|200.0|  c4|Chandelr|
|  c2|    Pasta|  5.0|  c2|    Joey|
|  c1|    Bread|  2.0|  c1|   Alice|
+----+---------+-----+----+--------+

